# Homework: Vector Search
## Q1. Embedding a query

In [1]:
from embedder import Embedder

embedder = Embedder()

2026-07-20 15:50:55.522903116 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
query = "How does approximate nearest neighbor search work?"
query_vector = embedder.encode(query)

In [3]:
query_vector[0]

np.float64(-0.02058203437252893)

## Q2. Cosine similarity

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
  repo_owner="DataTalksClub",
  repo_name="llm-zoomcamp",
	commit_id="8c1834d",
	allowed_extensions={"md"},
	filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
content = ""

for document in documents:
  if document['filename'] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
    content = document['content']
    break

In [6]:
document_vector = embedder.encode(content)

In [7]:
cosine_similarity = query_vector.dot(document_vector)
print(round(cosine_similarity, 2))

0.36


## Q3. Chunking and search by hand

In [8]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [9]:
import numpy as np
X = np.array(embedder.encode_batch([chunk['content'] for chunk in chunks]))

In [10]:
scores = X.dot(query_vector)

In [11]:
idx = np.argmax(scores)
chunks[idx]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

## Q4. Vector search with minsearch

In [16]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [17]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embedder.encode(query)

results = vindex.search(query_vector, num_results=5)

In [19]:
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search

In [23]:
from minsearch import Index

index = Index(
	text_fields=['content'],
	keyword_fields=['filename']
)

index.fit(chunks)

In [ ]:
question = "How do I store vectors in PostgreSQL?"

In [28]:
text_search_results = index.search(question, num_results=5)

In [29]:
query_vector = embedder.encode(question)
vector_search_results = vindex.search(query_vector, num_results=5)

In [30]:
[result['filename'] for result in text_search_results]

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [31]:
[result['filename'] for result in vector_search_results]

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']